## Cell 1 — Connect to Snowflake

In [1]:
import pyodbc
from pathlib import Path
import pandas as pd

cnxn = pyodbc.connect("DSN=HEMS")

InterfaceError: ('28000', '[28000] [Snowflake][DSI] An error occurred while attempting to retrieve the error message for key \'DSIReqSettingNotFound\' with message parameters [\'PWD\'] and component ID 2: Could not open error message files - Check that "C:\\Program Files\\SASHome\\AccessClients\\9.4\\Snowflake\\Bin\\en-US\\ODBCMessages.xml" or "C:\\Program Files\\SASHome\\AccessClients\\9.4\\Snowflake\\Bin\\ODBCMessages_en-US.xml" exists and are accessible. (-1) (SQLDriverConnect); [28000] [Snowflake][DSI] An error occurred while attempting to retrieve the error message for key \'DSIReqSettingNotFound\' with message parameters [\'PWD\'] and component ID 2: Could not open error message files - Check that "C:\\Program Files\\SASHome\\AccessClients\\9.4\\Snowflake\\Bin\\en-US\\ODBCMessages.xml" or "C:\\Program Files\\SASHome\\AccessClients\\9.4\\Snowflake\\Bin\\ODBCMessages_en-US.xml" exists and are accessible. (-1)')

## Cell 2 — Run Step 01 (create RAWT)

In [ ]:
sql = Path("sql/power60_allcirc/01_create_working_table_POWER60_RAWT.sql").read_text()
cnxn.cursor().execute(sql)
cnxn.commit()

## Cell 3 — Interactive inspection of flagged sites

In [ ]:
sql = Path("sql/power60_allcirc/02_list_flagged_sites_missing_or_invalid_mains.sql").read_text()

df_flagged = pd.read_sql(sql, cnxn)
df_flagged.head(20)

## Cell 4 — Lightweight diagnostics (ad hoc)

### Diagnostic 1

In [ ]:
df_flagged["flag_reason"].value_counts()

### Diagnostic 2

In [ ]:
df_flagged.query("mains_row_count > 0").head()

## Cell 5 — Explicit “go / no‑go” moment

Reviewed flagged sites and confirmed logic is acceptable. Proceeding to insert synthetic rows

## Cell 6 — Run Step 04 + 05 (map + insert)

In [ ]:
for step in [
    "04_create_point_map_9990x.sql",
    "05_insert_mains_all_circ_rows.sql"
]:
    sql = Path(f"sql/power60_allcirc/{step}").read_text()
    cnxn.cursor().execute(sql)
    cnxn.commit()

## Cell 7 — Post‑insert validation, shown inline

If anything returns rows → stop.

In [ ]:
sql = Path("sql/power60_allcirc/06_validate_added_rows.sql").read_text()
pd.read_sql(sql, cnxn)

In [ ]:
sql = Path("sql/power60_allcirc/07_validate_one_row_per_timestamp.sql").read_text()
pd.read_sql(sql, cnxn)